In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
import warnings
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from IPython.display import Image, display  # noqa: E402
from sklearn.decomposition import FastICA  # noqa: E402
from sklearn.exceptions import ConvergenceWarning  # noqa: E402

from scripts.notebook_helpers import (  # noqa: E402
    WAVELET_FREQ_MAX,
    WAVELET_FREQ_MIN,
    WAVELET_N_FREQS,
    load_paired_condition_wavelets,
    resolve_notebook_wavelet_cache_dir,
    resolve_wavelet_dir,
)
from src.analysis import iva_quality  # noqa: E402
from src.analysis.condition_tracks import ZSCORE_MODES  # noqa: E402
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    REAL_CONDITIONS,
    ConditionVariants,
    ExclusionCategories,
    ExperimentNames,
    MusicTypeVariants,
)
from src.visualization.iva_condition_plots import (  # noqa: E402
    plot_condition_mean_topomaps,
    plot_participant_condition_topomaps,
)
from src.visualization.iva_quality_plots import (  # noqa: E402
    save_fig,
    topo_info_subset,
)
from src.visualization.jica_plots import (  # noqa: E402
    DIFFERENCE_ROW,
    plot_global_tf_grid,
    plot_loading_bars,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
%matplotlib inline
print("Setup complete.")

# Joint ICA of Wavelet Power — Channel Components, Conditions Concatenated in Time

The **second** way to join the conditions under joint ICA, and the companion of
[`wavelet_ica_channel_joined.ipynb`](wavelet_ica_channel_joined.ipynb). Both run the
identical jICA pipeline on the identical participants; they differ only in **which axis
the two conditions are joined along**, and that one choice decides where a condition
difference can show up at all.

Here each participant contributes **one** channel block whose recording is their Placebo
track followed by their Psilocybin track
([`ConditionVariants.JOINED_TRACKS`](../../src/definitions/fields.py)):

```
participant k  ->  [ ---- Placebo track ---- | ---- Psilocybin track ---- ]
                   0                     T_pl                  T_pl + T_ps
```

```
Input:      (n_participants, n_channels, n_freqs, n_times_total)
Stack:      (n_participants × n_channels,  n_freqs × n_times_total)
            ──────── mixing ────────────   ──────── samples ───────
Transpose:  (n_freqs × n_times_total,  n_participants × n_channels)   ← FastICA input
FastICA over the stacked channel axis → N_ICA components
```

## What this buys, and what it costs

**The contrast lives in the time-frequency maps, and only there.** The sample axis now
spans both conditions, so each component's single global `(F, T_total)` map splits back
into a Placebo half and a Psilocybin half — a genuine per-condition TF contrast, which the
recording-axis variant cannot produce at all (there the sample axis is common to both
conditions by construction).

**The price is the mixing side.** One channel block per participant means **one topography
and one loading per (participant, component), shared by both conditions**. And that is
stronger than it is for IVA: because the sources are shared and unit-variance, a
participant's loading on a component is exactly proportional to the squared norm of its
pattern block, so restricting it to one condition's samples would only rescale every
participant by the *same* factor. There is therefore **no per-condition loading to
compare**, not even in principle. Step 5 and Step 6 draw the topographies and the loadings
once, labelled as shared, rather than two identical rows and a zero difference. For a
condition contrast **in the scalp pattern or the loading**, that is exactly what the
recording-axis variant is for.

|  | this notebook (`JOINED_TRACKS`) | [recording-axis](wavelet_ica_channel_joined.ipynb) (`JOINED`) |
|---|---|---|
| Joined along | **time** | the recording axis |
| Feature axis | `P × C` (each participant once) | `2P × C` (each participant twice) |
| Sources | one `(F, T_total)` map per component, split into the two conditions | one `(F, T)` map per component, shared |
| Topographies / loadings | one per participant, **shared** by the conditions | one per **recording** — the contrast |
| Contrast lives in | the **TF maps** | the **topographies and loadings** |

The two are complements, not alternatives: between them they cover both sides of the
decomposition, and neither can cover both on its own.

## Scope

Load, decompose, pin the sign and the order, split the time axis, and draw the component
**TF maps** per condition, the shared **topographies**, and the shared per-participant
**loading** as a bar plot. Nothing is scored, ranked or tested, and nothing is written to
disk beyond the figures and the reusable wavelet subset cache.

## Standardisation, and what a difference in Step 4 means

`ZSCORE_MODE` decides it, and it is the one setting to fix before reading any panel:

- `"per_condition"` (default) — each track is standardised on its own *before* the
  concatenation, so each enters the decomposition exactly as in a single-condition run and
  an overall power difference between the conditions is normalised away. A difference in
  Step 4 is then a difference in temporal and spectral **structure**.
- `"joint"` — concatenate first, standardise over the whole recording, so an overall power
  difference survives as a mean offset between the segments.

## How the data is assembled

`load_paired_condition_wavelets` loads each condition from its existing cache, keeps the
participants present in both, standardises each track and lays them end to end along time.
Nothing upstream changes: each condition keeps its own alignment, which is only ever used
*within* that condition, so the two tracks need not share a time base and need not even
have the same length. (That is where the two joins genuinely differ in requirements — the
recording-axis join *does* need one shared time base, because it stacks recordings that
must address the same samples.)

## Variables produced

| Variable | Shape | Description |
|----------|-------|-------------|
| `paired` | — | `PairedConditionTracks` — the concatenated tensor plus its segment bookkeeping |
| `bb_data` | `(P, C, F, T_total)` | Raw 4-D concatenated wavelet power |
| `ica_input` | `(F·T_total, P·C)` | Z-scored samples × stacked-channel matrix — the FastICA input |
| `tf_maps` | `(K, F, T_total)` | **Global** component TF maps, over both tracks |
| `tf_by_condition` | `(K, F, T_c)` each | The same maps split back into the two conditions |
| `patterns` | `(P, K, C)` | Forward (mixing) topographies — one per participant, **shared** |
| `ic_variance` | `(K,)` | Share of the joint channel-space energy each component accounts for |
| `subject_ic_energy` | `(P, K)` | That share split by participant — the shared loading |
| `retained` | float | Channel-space variance the whitening truncation kept |

## Configuration

In [ ]:
# ── Experiment configuration ───────────────────────────────────
# Choose the experiment: ExperimentNames.PSILO_MUSIC or ExperimentNames.ASSR
EXPERIMENT_NAME = ExperimentNames.ASSR
# The concatenated dataset IS the condition here — the two real conditions below are laid
# end to end along time, one channel block per participant.
CONDITION = ConditionVariants.JOINED_TRACKS
# Time-axis segment order: the Placebo track first, then Psilocybin.
CONDITIONS_TO_POOL = list(REAL_CONDITIONS)
if EXPERIMENT_NAME == ExperimentNames.ASSR:
    # ASSR has no music dimension; uses a single placeholder "music type".
    MUSIC_TYPE = MusicTypeVariants.ASSR
else:
    MUSIC_TYPE = MusicTypeVariants.CLASSICAL
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# ── Wavelet settings ─────────────────────────────────────────
REPRESENTATION = "power"
# Wavelet frequency grid: canonical 1 Hz-spaced grid from src.definitions.frequency
# (WAVELET_FREQ_MIN/MAX/N_FREQS), re-exported via scripts.notebook_helpers. It must
# match the grid the cache was written with, or the cache is missed and recomputed.
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

# ── Standardisation before the concatenation ─────────────────
# "per_condition" (default): standardise each track on its own, then concatenate — an
# overall power difference between the conditions is normalised away, so a difference in
# Step 4 is a difference in STRUCTURE. "joint" concatenates first and z-scores over the
# whole recording, keeping that difference as a mean offset between the segments. "none"
# assumes the caller already standardised.
ZSCORE_MODE = "per_condition"
assert ZSCORE_MODE in ZSCORE_MODES, f"ZSCORE_MODE must be one of {ZSCORE_MODES}"

# ── Reuse / compute ──────────────────────────────────────────
# True is the intended setting: this notebook is built around reusing the two
# per-condition caches. False would recompute both from RAW_AFTER_ICA.
REUSE_WAVELETS = True

# ── Cohort, channel and time subset ────────────────────────────
# N_PAIRS_SUBSET counts participants, which here is also the number of channel blocks:
# each participant carries both tracks.
N_PAIRS_SUBSET: int | None = 5
# Memory: the FastICA input is (F·T_total) × (P·C) float64, and T_total spans BOTH
# conditions — so the sample axis is twice a single condition's on top of the n_freqs
# factor. N_TIMES_SUBSET is applied per condition (per segment), so the concatenated axis
# is 2 × it. With the defaults below the input is 50·6000 × 5·32 ≈ 384 MB; the z-scored
# tensor it is built from is another 384 MB (released straight after), and FastICA's
# whitening SVD works on a transposed copy, so budget roughly three times the input for
# the fit.
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 3000  # first N samples OF EACH CONDITION

# ── ICA settings ──────────────────────────────────────────────
# N_ICA is the ONLY dimensionality knob: FastICA whitens the stacked channel matrix by
# SVD and keeps its leading N_ICA directions, so it performs the reduction itself and no
# separate PCA step is needed. It is NOT capped by the channel count — the feature axis
# is P·C — but it is capped in practice by convergence, which Step 2 reports.
N_ICA = 10
ICA_ALGORITHM = "parallel"  # or "deflation" — extracts one component at a time
ICA_FUN = "logcosh"  # contrast function: "logcosh", "exp" or "cube"
ICA_MAX_ITER = 2000
ICA_TOL = 1e-4
RANDOM_STATE = 42  # FastICA's initial unmixing is random; this pins it

# ── Figures ───────────────────────────────────────────────────
SAVE_PLOTS = True
# Which components the figures cover. None = every component.
COMPONENTS_TO_PLOT: list[int] | None = None
# Per-participant topography grids are one figure PER COMPONENT, so they are opt-in.
WRITE_PARTICIPANT_GRIDS = True
# Reference lines on the TF panels. The ASSR is continuous 40 Hz stimulation, so the
# stimulation frequency is the row worth locating on every map.
TF_FREQ_MARKS: list[float] = [iva_quality.ASSR_FREQ]
MARK_STIMULUS_ONSETS_ON_TF = True

# ── Stimulus-locked epoch (Step 7) ────────────────────────────
# The paradigm window from src.definitions.constants.AssrEpoch, so this notebook, the
# recording-axis variant, the 06 IVA notebooks and the 05 quality workflow all cut the
# SAME epoch. Applied to each condition's own split track, with its own onsets.
EPOCH_PRE_S = AssrEpoch.PRE_ONSET_S
EPOCH_POST_S = AssrEpoch.POST_ONSET_S
MIN_ONSETS_FOR_EPOCH_AVERAGE = 5

# ── Wavelet cache directories ─────────────────────────────────
# WAVELET_DIR is the source of truth (one big compressed file per condition);
# WAVELET_SUBSET_CACHE_DIR holds per-extent copies under the stage-03 notebook, shared
# with every other wavelet workflow. See the 03 README for the full comparison.
WAVELET_DIR: Path = resolve_wavelet_dir(None, EXPERIMENT_NAME) / "broadband"
WAVELET_SUBSET_CACHE_DIR: Path = (
    resolve_notebook_wavelet_cache_dir(EXPERIMENT_NAME) / "broadband"
)
REUSE_WAVELET_SUBSET_CACHE = True

# ── Plots directory ───────────────────────────────────────────
# Same stage as the recording-axis variant, but its own analysis_type subdirectory so the
# two sets of figures never collide.
PLOTS_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR
    / "07-ica-condition-comparison"
    / "plots"
    / EXPERIMENT_NAME.value
    / "broadband"
    / "ica_channel_joined_tracks"
    / f"ica_{N_ICA}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment / group  : {EXPERIMENT_NAME.value} — {CONDITION.value}")
print(f"Segment order       : {[c.value for c in CONDITIONS_TO_POOL]} (time axis)")
print(f"Standardisation     : {ZSCORE_MODE} (before the concatenation)")
print(f"Wavelet source cache: {WAVELET_DIR}")
print(
    f"Wavelet subset cache: {WAVELET_SUBSET_CACHE_DIR}  "
    f"(reuse: {REUSE_WAVELET_SUBSET_CACHE})"
)
print(f"Frequencies         : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)")
print(
    f"Decomposition       : {N_ICA} ICs over {ICA_ALGORITHM}/{ICA_FUN} "
    f"(FastICA whitens the stacked channel axis itself; subset extents only), "
    f"seed {RANDOM_STATE}"
)
print(f"Plots directory     : {PLOTS_DIR}  (saving: {SAVE_PLOTS})")
print(
    f"Stimulus epoch      : [-{EPOCH_PRE_S}, {EPOCH_POST_S}] s around each onset "
    f"({AssrEpoch.STIMULUS_DURATION_S} s stimulus + "
    f"{AssrEpoch.POST_STIMULUS_S} s post-stimulus)"
)

## Data Loading — tracks concatenated per participant

One call does the whole join: load each condition's wavelet power from its existing cache,
keep the participants present in both, standardise each track, and lay them end to end
along time.

**Memory note.** A cached wavelet tensor is decompressed in full before it is trimmed to
the requested channel/time extent, and the two conditions are loaded and trimmed **one at
a time**, so the peak is one condition's cache — the same peak as a single-condition
notebook. The trimmed result is then written to `WAVELET_SUBSET_CACHE_DIR`, so later runs
at the same extent skip that read entirely. Those cache entries are the same ones the
recording-axis notebook and the 03/04/05/06 workflows read.

In [ ]:
paired, condition_analyzers = load_paired_condition_wavelets(
    MUSIC_TYPE,
    EXCLUSION_CATEGORIES,
    FREQS,
    wavelet_dir=WAVELET_DIR,
    experiment_name=EXPERIMENT_NAME,
    representation=REPRESENTATION,
    conditions=CONDITIONS_TO_POOL,
    zscore_mode=ZSCORE_MODE,
    n_channels=N_CHANNELS_SUBSET,
    n_times=N_TIMES_SUBSET,
    reuse_wavelets=REUSE_WAVELETS,
    subset_cache_dir=WAVELET_SUBSET_CACHE_DIR,
    reuse_subset_cache=REUSE_WAVELET_SUBSET_CACHE,
)

print(f"Concatenated dataset : {paired.data.label}")
print(
    f"Participants         : {paired.n_pairs}  (each carries both tracks in one block)"
)
print(f"Shape                : {paired.data.data.shape}")
print(
    f"Segment lengths      : {paired.segment_lengths}  -> total {paired.total_length}"
)
print(f"Segment boundaries   : {paired.boundaries}")

# Restrict the cohort by participant. Here that is also the block axis, so a leading slice
# would be correct — but going through the participant labels keeps this cell identical in
# meaning to the recording-axis notebook, where it would not be.
if N_PAIRS_SUBSET is not None and N_PAIRS_SUBSET < paired.n_pairs:
    keep = sorted(paired.participants)[:N_PAIRS_SUBSET]
    rows = [paired.participants.index(p) for p in keep]
    paired = dataclasses.replace(
        paired,
        data=dataclasses.replace(paired.data, data=paired.data.data[rows]),
        participants=tuple(keep),
    )
    print(f"\nUsing first {len(keep)} participant(s): {keep}")
    print(f"Shape                : {paired.data.data.shape}")

print(f"\nParticipants         : {list(paired.participants)}")

## Dataset Selection

Unpacks the concatenated dataset into the names the other variants use (`bb_data`,
`sfreq`, …), records the segment bookkeeping the split in Step 4 needs, builds the topomap
`Info`, and names the two shared figures this notebook uses beyond
[`src/visualization/iva_condition_plots.py`](../../src/visualization/iva_condition_plots.py):

Both come from
[`src/visualization/jica_plots.py`](../../src/visualization/jica_plots.py) rather than
being defined here, so this notebook and
[`scripts/run_wavelet_jica.py`](../../scripts/run_wavelet_jica.py) draw the identical
figure from the identical code.

- `plot_global_tf_grid` — the component TF maps. Deliberately **not**
  `plot_condition_mean_tf_maps`: there is one map per component here rather than one per
  recording, so there is no recording axis to average over — and that function's first
  step is `equalize_subject_influence`, which with one map per row would rescale each row
  by its own amplitude and normalise away exactly the between-condition difference the
  grid is drawn to show.
- `plot_loading_bars` — the per-participant loading.

The per-condition stimulus onsets are kept **local to their own segment**, which is what
`condition_track` returns, so they line up with the split tracks without shifting.

In [ ]:
LABEL = paired.data.label

bb_ad = paired.data
bb_data = bb_ad.data  # (n_participants, n_channels, n_freqs, n_times_total)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times_total = bb_data.shape
time_total = np.arange(n_times_total) / sfreq

participants = list(paired.participants)
CONDITION_ROWS = [c.value for c in paired.conditions]
# Row label for anything the conditions SHARE — here the topographies and the loadings.
SHARED_ROW = " + ".join(CONDITION_ROWS) + " (shared)"
# What pins the component sign, named on every figure. One sign per component, not per
# participant: jICA has no per-recording sign ambiguity to resolve.
SIGN_NOTE = "largest |TF| excursion positive (one sign per component)"

# Per-condition onsets, local to that condition's own segment.
onsets_by_condition = {
    c: paired.condition_onsets(c, absolute=False) for c in paired.conditions
}

print(f"Dataset      : {LABEL}")
print(f"Shape        : {bb_data.shape}  (participants × channels × freqs × times)")
print(f"Duration     : {n_times_total / sfreq:.1f} s total  @  {sfreq} Hz")
print(f"Freq range   : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs} steps)")
print(f"Participants : {n_subjects}  (one channel block each)")
for condition in paired.conditions:
    segment = paired.segment(condition)
    onsets = onsets_by_condition[condition]
    print(
        f"  {condition.value:<12}: samples [{segment.start}, {segment.stop}) "
        f"= {(segment.stop - segment.start) / sfreq:.1f} s, "
        f"{0 if onsets is None else len(onsets)} onset(s)"
    )

# ── Topomap layout ───────────────────────────────────────────
# The analyser's Info, restricted to the notebook's channel subset in the same order as
# the decomposition's channel axis. Both conditions were preprocessed onto the same
# montage, so either analyser will do.
ica_info = topo_info_subset(condition_analyzers[paired.conditions[0]].info, n_channels)
print(
    f"Topomap channels : {len(ica_info['ch_names'])} "
    f"(first={ica_info['ch_names'][0]}, last={ica_info['ch_names'][-1]})"
)


# ── Figure helpers ───────────────────────────────────────────
# The two figures this notebook needs beyond
# src/visualization/iva_condition_plots.py live in
# src/visualization/jica_plots.py, imported above rather than defined here so the
# notebook and scripts/run_wavelet_jica.py draw the identical figure:
#
#   plot_global_tf_grid  the component TF maps. Deliberately NOT
#       plot_condition_mean_tf_maps: there is one map per component here, shared by
#       every recording, so there is no recording axis to average over — and that
#       function's first step is equalize_subject_influence, which with one map per
#       row would rescale each row by its own amplitude and normalise away exactly
#       the between-row difference a grid is drawn to show.
#   plot_loading_bars    the per-recording loading, grouped by participant.
#
# DIFFERENCE_ROW is the row label that gets its own colour limit, since a
# difference is far weaker than the maps it comes from.


def _save(fig, name: str) -> None:
    """Write *fig* into PLOTS_DIR under the band prefix, when saving is on."""
    if not SAVE_PLOTS:
        return
    path = PLOTS_DIR / f"{name}.png"
    save_fig(fig, path)
    print(f"saved {path}")

---
## Step 1 — Z-score, Then Lay the Participants' Channel Axes Side by Side

Applied for symmetry with the other variants, so this notebook runs the identical
pipeline. With `ZSCORE_MODE = "per_condition"` the z-scoring here is an **exact
identity**: each track was already standardised before the concatenation, and two
unit-variance zero-mean segments concatenate to a series with mean 0 and variance 1. The
cell checks that numerically rather than asserting it in prose — under `"joint"` or
`"none"` it is a real normalisation and the printed deviation will be non-zero.

**The stack.** Each participant's `(C, F, T_total)` slice is flattened to
`(C, F·T_total)`, the `P` blocks are stacked into a single `(P·C, F·T_total)` matrix, and
the result is transposed so FastICA sees `(F·T_total, P·C)`: **samples = (frequency, time)
bins spanning both conditions, mixing variables = every participant's channels**. The
flatten keeps frequency slow and time fast (`index = f·T + t`), so the sample axis
reshapes cleanly back to `(F, T_total)` after the decomposition — and only then is it safe
to split. The feature index is `p·C + c`, so the mixing column reshapes cleanly back to
`(P, C)`.

Every column of the stacked matrix has unit variance by construction, which is what makes
the stacking fair: no participant can dominate the joint whitening by amplitude.

In [ ]:
bb_z = zscore_by_time(bb_data)  # (P, C, F, T_total)

# With "per_condition" this changed nothing; say so with a number, not a claim.
_max_dev = float(np.abs(bb_z - bb_data).max())
_scale = float(np.abs(bb_data).max())
print(f"z-score max |change| : {_max_dev:.3e}  (data |max| {_scale:.3g})")
if ZSCORE_MODE == "per_condition":
    print(
        "  -> expected ~0: each track was standardised before the concatenation, so the "
        "concatenated series already has mean 0 and unit variance."
    )
    assert _max_dev < 1e-8 * max(_scale, 1.0), (
        "zscore_by_time was expected to be an identity under 'per_condition' but "
        f"changed the data by {_max_dev:.3e}."
    )

# Stack: (P, C, F, T) → (P*C, F*T) → transpose to samples × mixing.
n_samples_ft = n_freqs * n_times_total
n_features = n_subjects * n_channels
ica_input = np.ascontiguousarray(bb_z.reshape(n_features, n_samples_ft).T)

print(f"\nStacked reshape : {ica_input.shape}  (F*T samples, P*C mixing variables)")
print(f"  Samples       : {n_samples_ft}  (F={n_freqs} * T_total={n_times_total})")
print(f"  Mixing dim    : {n_features}  (P={n_subjects} participants * C={n_channels})")
print(f"  Memory        : {ica_input.nbytes / 1e9:.2f} GB")

# Every column must have unit variance — that is what makes the stacking fair, so it is
# checked rather than claimed.
_col_std = ica_input.std(axis=0)
print(
    f"\nColumn std      : min {_col_std.min():.4f}, max {_col_std.max():.4f} "
    "(1.0 = no participant can dominate the whitening by gain)"
)
_worst = float(np.abs(_col_std - 1.0).max())
if _worst > 1e-6:
    print(
        f"  NOTE: worst deviation {_worst:.2e} — some (participant, channel, frequency) "
        "series was constant in time and left at its guard value."
    )

# The z-scored tensor is only needed to build the matrix; release it (bb_data, the raw
# concatenated tensor, stays alive inside ``paired``).
del bb_z

---
## Step 2 — One Joint FastICA Over the Stacked Channel Axis

Identical to the recording-axis variant, and deliberately so: only the matrix differs.
FastICA unmixes the stacked channel axis directly, and because the sample axis is the
joint (frequency × time) plane, the components come out as spectro-temporal sources —
spanning **both** conditions at this point. The split comes in Step 4, after the sign is
settled.

**`FastICA` does the channel reduction itself here, and that is why this notebook is
capped to a subset extent.** `FastICA(n_components=K)` whitens the matrix by SVD and keeps
its leading `K` directions — the same subspace a `PCA(K)` would have produced — but it
gets there by calling `scipy.linalg.svd`, and LAPACK indexes with **32-bit integers**. A
matrix with more than `2**31 - 1` elements is refused outright:

```
ValueError: Indexing a matrix of 10993320000 elements would incur an in integer
overflow in LAPACK.
```

At the full ASSR extent (12 participants × 195 channels = 2340 features) that ceiling is
~918k samples, i.e. ~73 s of both tracks on a 50-bin frequency grid. This is a hard limit,
not a memory budget: no node size makes the call work.

The cell below therefore **refuses** rather than failing deep inside sklearn, and says
where to go instead. For the full recording use
[`scripts/run_wavelet_jica.py`](../../scripts/run_wavelet_jica.py), whose
[`fit_joint_ica`](../../src/analysis/wavelet_jica.py) takes the leading directions from the
`(features, features)` covariance — 2340 × 2340, trivial — and runs FastICA on the scores.
The two routes span the *same* subspace (verified equal to 4e-16), so this notebook and
that script describe the same decomposition; only the arithmetic that gets there differs.

(The IVA variants *do* need their explicit per-recording PCA, for a different reason:
`iva_g` requires a square mixing matrix per dataset.)

Four outputs deserve attention.

**`tf_maps`** `(K, F, T_total)` — each component's score map over the whole concatenated
recording. `unit-variance` whitening fixes every source to unit variance, so the maps
share a scale; the amplitude lives in the pattern instead.

**`patterns`** `(P, K, C)` — the **forward (mixing)** channel patterns, `ica.mixing_.T`
reshaped to split the stacked feature axis back into participants. This is what belongs on
a topomap; the unmixing rows (`components_`) are spatial *filters*, and plotting those
instead is the classic filter-vs-pattern error (Haufe et al., 2014, NeuroImage 87:96-110).
**One topography per (participant, component), shared by both conditions** — that is this
variant's construction.

**`retained`** — the fraction of the joint channel-space variance the whitening truncation
kept: the hard ceiling on everything the components can carry.

**`ic_variance`** `(K,)` and **`subject_ic_energy`** `(P, K)` — the energy of each
component's rank-one back-projection over the total sum of squares, and that energy split
by participant. The split is exact because the feature axis partitions by participant.

In [ ]:
def fit_joint_channel_ica(matrix, n_ica, seed, chunk=20_000):
    """Fit one FastICA over the stacked channel axis of the concatenated tensor.

    :param matrix: ``(F*T, P*C)`` samples × stacked-channels matrix.
    :param n_ica: Independent components to extract. Also sets the whitening
        truncation, since FastICA reduces the feature axis itself.
    :raises ValueError: If *matrix* has more elements than 32-bit LAPACK can index,
        which is a hard ceiling rather than a memory limit — see the markdown above
        and use scripts/run_wavelet_jica.py for the full extent.
    :param seed: ``random_state`` for FastICA's initial unmixing.
    :param chunk: Sample rows per block when the sums of squares are accumulated. The
        centred matrix and the back-projection are each the size of *matrix*, so
        forming them whole would triple the peak for two scalars; chunking keeps it at
        a few tens of MB and gives the identical result.
    :return: ``(sources, patterns_stacked, ic_variance, feature_shares, retained,
        total_ss, converged)`` with shapes ``(F*T, K)``, ``(K, P*C)``, ``(K,)``,
        ``(K, P*C)``, float, float, bool. *feature_shares* is each feature's rank-one
        energy share, ready to be pooled per participant.
    """
    # scipy.linalg.svd, which FastICA's whitening calls, indexes with 32-bit ints.
    # Refuse here with something actionable rather than failing deep inside sklearn.
    if matrix.size > np.iinfo(np.int32).max:
        raise ValueError(
            f"The stacked matrix has {matrix.size:,} elements, above the "
            f"{np.iinfo(np.int32).max:,} that 32-bit LAPACK can index, so FastICA's "
            "whitening SVD cannot run on it at any node size. Lower N_TIMES_SUBSET / "
            "N_CHANNELS_SUBSET / N_PAIRS_SUBSET for this notebook, or use "
            "scripts/run_wavelet_jica.py, which reduces the channel axis through the "
            "(features, features) covariance instead and has no such ceiling."
        )
    ica = FastICA(
        n_components=n_ica,
        algorithm=ICA_ALGORITHM,
        fun=ICA_FUN,
        whiten="unit-variance",
        random_state=seed,
        max_iter=ICA_MAX_ITER,
        tol=ICA_TOL,
    )
    # Capture ConvergenceWarning instead of letting it print once and vanish.
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        sources = ica.fit_transform(matrix)  # (F*T, K)
    converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)

    # Forward model, directly in stacked-channel space.
    patterns_stacked = ica.mixing_.T  # (K, P*C)

    # Total and residual sums of squares, accumulated in blocks of samples so neither
    # the centred matrix nor the back-projection is ever formed whole.
    feature_mean = matrix.mean(axis=0)
    total_ss = 0.0
    residual_ss = 0.0
    for start in range(0, matrix.shape[0], chunk):
        block = matrix[start : start + chunk] - feature_mean
        total_ss += float((block**2).sum())
        block -= sources[start : start + chunk] @ patterns_stacked
        residual_ss += float((block**2).sum())
    if total_ss <= 0.0:
        nan_k = np.full(n_ica, np.nan)
        return (
            sources,
            patterns_stacked,
            nan_k,
            np.full_like(patterns_stacked, np.nan),
            np.nan,
            total_ss,
            converged,
        )
    retained = 1.0 - residual_ss / total_ss
    source_ss = (sources**2).sum(axis=0)  # (K,)
    # Rank-one energy, per feature. Summing over a participant's channel block gives
    # that participant's share; summing over everything gives ic_variance.
    feature_shares = source_ss[:, np.newaxis] * patterns_stacked**2 / total_ss
    ic_variance = feature_shares.sum(axis=1)
    return (
        sources,
        patterns_stacked,
        ic_variance,
        feature_shares,
        retained,
        total_ss,
        converged,
    )


(
    sources_ft,
    patterns_stacked,
    ic_variance,
    _feature_shares,
    retained,
    total_ss,
    converged,
) = fit_joint_channel_ica(ica_input, N_ICA, RANDOM_STATE)

# Fold both axes back: the samples into (F, T_total), the stacked features into (P, C).
tf_maps = sources_ft.T.reshape(N_ICA, n_freqs, n_times_total)  # (K, F, T_total)
patterns = patterns_stacked.reshape(N_ICA, n_subjects, n_channels).transpose(1, 0, 2)
subject_ic_energy = (
    _feature_shares.reshape(N_ICA, n_subjects, n_channels).sum(axis=2).T
)  # (P, K)

print(f"Sources (F*T, K)      : {sources_ft.shape}")
print(f"TF maps (K, F, T_tot) : {tf_maps.shape}  — spanning BOTH conditions")
print(f"Patterns (P, K, C)    : {patterns.shape}")
print("  ^ one topography per (participant, component) — SHARED by both conditions")
print(f"Loading (P, K)        : {subject_ic_energy.shape}  — also shared")
print(
    f"\nRetained variance     : {retained * 100:.1f}% of the joint channel space "
    f"({N_ICA} of {n_features} directions)"
)
print(
    f"ic_variance           : top {ic_variance.max() * 100:.2f}%, "
    f"sum {ic_variance.sum() * 100:.2f}%"
)
print(f"FastICA converged     : {converged} ({ICA_ALGORITHM}/{ICA_FUN})")
if not converged:
    print(
        f"  WARNING: FastICA did not converge within {ICA_MAX_ITER} iterations at "
        f"tol={ICA_TOL:g}, so the unmixing is wherever the solver stopped rather than a "
        "fixed point. Lower N_ICA, or try ICA_ALGORITHM='deflation'. On ASSR data "
        "raising ICA_MAX_ITER tends to make the result drift further rather than settle."
    )

# The per-participant split must add up to the component's total, by construction — a
# mismatch would mean the feature axis was folded back in the wrong order.
np.testing.assert_allclose(
    subject_ic_energy.sum(axis=0),
    ic_variance,
    rtol=1e-9,
    atol=1e-12,
    err_msg="The per-participant energy split does not sum to ic_variance; the stacked "
    "feature axis was reshaped in the wrong order.",
)
print("\nPer-participant energy split sums to ic_variance.")

---
## Step 3 — Sign and Order

FastICA fixes neither the sign nor the order of its components, so both are pinned before
plotting — otherwise the same data replotted with a different seed looks like a different
result. Neither choice is a claim about the data: the sign flips each component so the
**largest absolute excursion of its TF map is positive** (hypothesis-free — it says
nothing about 40 Hz), and the order sorts by `ic_variance` descending, which is a statement
about *size*, not relevance.

**The sign is settled before the split, and that matters here.** A sign belongs to a
component, and it multiplies the map and the entire mixing column together. Flipping a
component's two time segments differently would destroy the one property this variant is
built on — that the component is the same in both conditions — and there is no mechanism
here that could: one sign per component means both halves inherit it. (In the IVA variants
the sign belongs to a `(recording, component)` pair, which is why they spend two alignment
passes on it.)

Both operations leave the back-projection `Σ_k source_k ⊗ pattern_k` untouched, which is
what the assertion checks on a random sample of rows.

In [ ]:
def back_projection(tf, pat, rows):
    """``Σ_k source_k ⊗ pattern_k`` on selected sample rows — sign- and order-invariant.

    Only *rows* are reconstructed: the full back-projection is the size of the whole
    input matrix, and a random few thousand samples settle the question just as well.

    :param tf: ``(K, F, T)`` global component maps.
    :param pat: ``(P, K, C)`` per-participant forward patterns.
    :param rows: Sample indices into the flattened ``F*T`` axis.
    :return: ``(len(rows), P*C)`` reconstruction of those rows of the stacked matrix.
    """
    n_k = tf.shape[0]
    return tf.reshape(n_k, -1)[:, rows].T @ pat.transpose(1, 0, 2).reshape(n_k, -1)


_check_rows = np.random.default_rng(RANDOM_STATE).choice(
    n_samples_ft, size=min(2000, n_samples_ft), replace=False
)
_recon_before = back_projection(tf_maps, patterns, _check_rows)

# ── Sign: make each component's largest absolute TF excursion positive ─────────
_flat = tf_maps.reshape(N_ICA, -1)
_peak = np.take_along_axis(_flat, np.abs(_flat).argmax(axis=1)[:, np.newaxis], axis=1)
signs = np.where(_peak[:, 0] < 0.0, -1.0, 1.0)  # (K,)
tf_maps = tf_maps * signs[:, np.newaxis, np.newaxis]
patterns = patterns * signs[np.newaxis, :, np.newaxis]

# ── Order: largest share of the joint channel-space energy first ───────────────
order = np.argsort(-ic_variance, kind="stable")
tf_maps = tf_maps[order]
patterns = patterns[:, order]
ic_variance = ic_variance[order]
subject_ic_energy = subject_ic_energy[:, order]
sources_ft = sources_ft[:, order] * signs[order][np.newaxis, :]

# Neither the flip nor the reordering may change what the components add up to.
np.testing.assert_allclose(
    back_projection(tf_maps, patterns, _check_rows),
    _recon_before,
    atol=1e-9,
    err_msg="Re-orienting or reordering changed the back-projection — the sign was not "
    "applied to the map and the patterns together.",
)
del _recon_before

COMP_INDICES = list(range(N_ICA)) if COMPONENTS_TO_PLOT is None else COMPONENTS_TO_PLOT
_out_of_range = [k + 1 for k in COMP_INDICES if not 0 <= k < N_ICA]
if _out_of_range:
    raise ValueError(
        f"COMPONENTS_TO_PLOT names IC {_out_of_range}, outside 1..{N_ICA}."
    )

print(
    f"Flipped {int((signs < 0).sum())}/{signs.size} component(s); reordered by "
    "ic_variance. Back-projection unchanged."
)
print("\n  IC   ic_variance   loading spread across participants (min–max % of IC)")
for k in range(N_ICA):
    _share = subject_ic_energy[:, k] / subject_ic_energy[:, k].sum()
    print(
        f"  {k + 1:>3}   {ic_variance[k] * 100:9.2f}%   "
        f"{_share.min() * 100:5.1f} – {_share.max() * 100:5.1f}   "
        f"(even split would be {100 / n_subjects:.1f})"
    )
print(f"\nComponents in the figures: {[k + 1 for k in COMP_INDICES]}")

---
## Step 4 — Split the Time Axis Back Into the Two Conditions

The payoff of the whole construction, and it is one slice.
`paired.condition_track(array, condition)` cuts any array whose **last** axis is the
concatenated time axis, so it works unchanged on the global maps `(K, F, T_total)`. It
validates that last axis, so passing something that did not come from this concatenation
raises rather than silently mis-cutting — which is also why the channel patterns `(P, K,
C)` cannot be passed through it by accident.

Rows = conditions plus their **difference** (Psilocybin − Placebo), columns = components.
Same scaling rules as everywhere: one symmetric limit per column shared by the two
condition rows, no limit shared across columns, the difference row on its own limit.

**No per-recording equalisation, deliberately.** There is one map per condition here, not
an average over recordings, so there is no per-recording gain to remove — and rescaling
each row by its own amplitude, which is what `plot_condition_mean_tf_maps` would do, would
normalise away exactly the difference the grid exists to show.

**What a difference means** depends on `ZSCORE_MODE`, which the cell prints: under
`"per_condition"` each track was standardised on its own, so an overall power difference
was normalised away and what is left is temporal and spectral *structure*. The conditions
keep their own time bases and may differ in length; the grid needs one axis, so it is drawn
on the shorter one, and each condition's full-length track stays available in
`tf_by_condition`.

In [ ]:
tf_by_condition = {
    c.value: paired.condition_track(tf_maps, c) for c in paired.conditions
}
for name, arr in tf_by_condition.items():
    print(f"{name:<12}: {arr.shape}  (K, F, T_condition)")

# One common axis for the grid: the shorter segment. Full-length tracks stay in
# tf_by_condition.
n_times_split = min(arr.shape[-1] for arr in tf_by_condition.values())
maps_by_row = {name: arr[..., :n_times_split] for name, arr in tf_by_condition.items()}
maps_by_row[DIFFERENCE_ROW] = (
    maps_by_row[CONDITION_ROWS[1]] - maps_by_row[CONDITION_ROWS[0]]
)
time_split = np.arange(n_times_split) / sfreq

_lengths = {name: arr.shape[-1] for name, arr in tf_by_condition.items()}
if len(set(_lengths.values())) > 1:
    print(
        f"\n  NOTE: segments differ in length {_lengths}; the grid is drawn on "
        f"{n_times_split} samples. Read tf_by_condition for the full tracks."
    )
print(f"\nz-score mode : {ZSCORE_MODE}")
if ZSCORE_MODE == "per_condition":
    print(
        "  Each track was standardised on its own, so an overall power difference "
        "between\n  the conditions was normalised away. The difference row is a "
        "difference in temporal\n  and spectral STRUCTURE, not in amplitude."
    )
else:
    print(
        "  Standardised over the whole concatenated recording, so an overall power "
        "difference\n  between the conditions survives as an offset between the "
        "segments."
    )

# Onsets on each condition's own (trimmed) split axis, for the faint TF markers.
_ref_onsets = onsets_by_condition[paired.conditions[0]]
stimulus_onset_times = (
    np.array([])
    if _ref_onsets is None
    else _ref_onsets[_ref_onsets < n_times_split] / sfreq
)
print(f"Stimulus onsets in the split window : {len(stimulus_onset_times)}")

fig_tf = plot_global_tf_grid(
    maps_by_row,
    [*CONDITION_ROWS, DIFFERENCE_ROW],
    COMP_INDICES,
    FREQS,
    time_split,
    label=LABEL,
    title="Component TF maps, split by condition",
    time_marks=stimulus_onset_times if MARK_STIMULUS_ONSETS_ON_TF else None,
    freq_marks=TF_FREQ_MARKS,
    sign_note=SIGN_NOTE,
    save_path=(PLOTS_DIR / "condition_tf_maps.png") if SAVE_PLOTS else None,
)
plt.show()
plt.close("all")
if SAVE_PLOTS:
    print(f"Saved to {PLOTS_DIR}")

---
## Step 5 — The Shared Component Topographies

`patterns` holds **one** topography per `(participant, component)`, shared by both
conditions because there is one channel block per participant. So the grid is drawn with a
**single row**, labelled accordingly. Drawing it as a two-row condition comparison would
produce two identical rows and an all-zero difference — a figure that looks like a null
result but is really a statement about the model, not about psilocybin.

For a genuine topography contrast, use
[`wavelet_ica_channel_joined.ipynb`](wavelet_ica_channel_joined.ipynb), where each
recording has its own block of the mixing column.

Every participant is put on a common scale first (`equalize_subject_influence`), so the
row is a comparison of topographic *shape* across participants rather than a report on the
loudest few; the amplitude question is Step 6's.

In [ ]:
fig_topo = plot_condition_mean_topomaps(
    patterns,
    participants,
    [SHARED_ROW] * n_subjects,
    [SHARED_ROW],
    ica_info,
    n_channels,
    COMP_INDICES,
    label=LABEL,
    alignment_note=SIGN_NOTE,
    save_path=(PLOTS_DIR / "shared_mean_topomaps.png") if SAVE_PLOTS else None,
)
plt.show()
plt.close("all")

if WRITE_PARTICIPANT_GRIDS:
    topo_paths = plot_participant_condition_topomaps(
        patterns,
        participants,
        [SHARED_ROW] * n_subjects,
        [SHARED_ROW],
        ica_info,
        n_channels,
        COMP_INDICES,
        label=LABEL,
        root_dir=PLOTS_DIR / "participants",
        prefix="shared_",
        alignment_note=SIGN_NOTE,
    )
    print(
        f"Wrote {len(topo_paths)} shared-topography figure(s) to "
        f"{PLOTS_DIR / 'participants'}"
    )
    for path in topo_paths:
        display(Image(filename=str(path)))

---
## Step 6 — Each Participant's Contribution to the Loading

One bar per participant per component, and — like the topographies — **shared by both
conditions**. It is worth being precise about why there is no condition split here, because
it is a stronger statement than "we did not compute one":

the sources are shared and unit-variance, so a participant's rank-one energy on component
*k* is exactly `‖pattern block‖²` times a factor common to every participant. Restricting
that to one condition's samples multiplies every participant by the *same* number, so a
per-condition loading in this variant carries no participant-level information that the
shared one does not. The condition contrast lives entirely in the source split (Step 4).

So this figure answers a different question, and a necessary one: **who drives each
component**. Two normalisations of the same quantity:

- **Share of the joint channel-space energy** — participant *p*'s rank-one energy on
  component *k* as a percentage of the total sum of squares. Comparable across components
  and participants.
- **Share of the component** (each column normalised to 100%) — how the component's energy
  is divided among the participants. An even split over `P` participants would be `100/P` %
  each; a component sitting almost entirely on one or two bars is not a group result, and a
  condition difference drawn from its TF maps in Step 4 should be read with that in mind.

A bar is a *magnitude*: a participant whose topography is inverted relative to the group
still gets a tall bar, so read this alongside Step 5.

In [ ]:
_shared_rows = [SHARED_ROW] * n_subjects

# ── Absolute: share of the joint channel-space energy ─────────────────────────
fig_load = plot_loading_bars(
    subject_ic_energy * 100.0,
    participants,
    _shared_rows,
    [SHARED_ROW],
    COMP_INDICES,
    label=LABEL,
    title="Per-participant component loading (shared by both conditions)",
    ylabel="% of joint channel-space energy",
    save_path=(PLOTS_DIR / "loading_bars_absolute.png") if SAVE_PLOTS else None,
)
plt.show()
plt.close("all")

# ── Relative: how each component's energy is divided among the participants ───
within_component = subject_ic_energy / subject_ic_energy.sum(axis=0, keepdims=True)
fig_load_rel = plot_loading_bars(
    within_component * 100.0,
    participants,
    _shared_rows,
    [SHARED_ROW],
    COMP_INDICES,
    label=LABEL,
    title=(
        f"Share of each component carried by each participant "
        f"(even split = {100 / n_subjects:.1f}%)"
    ),
    ylabel="% of this IC's energy",
    save_path=(PLOTS_DIR / "loading_bars_within_component.png") if SAVE_PLOTS else None,
)
plt.show()
plt.close("all")

# ── The same numbers as a table ───────────────────────────────────────────────
loading_frame = pd.DataFrame(
    {
        "IC": np.repeat([k + 1 for k in COMP_INDICES], n_subjects),
        "participant": participants * len(COMP_INDICES),
        "energy_pct": np.concatenate(
            [subject_ic_energy[:, k] * 100.0 for k in COMP_INDICES]
        ),
        "within_ic_pct": np.concatenate(
            [within_component[:, k] * 100.0 for k in COMP_INDICES]
        ),
    }
)
print("Loading table: % of joint channel-space energy, and % of each IC's own energy")
display(loading_frame.round(3))

---
## Step 7 — Stimulus-Locked TF Comparison (Averaged Over Stimuli)

The third view, as in the IVA variants: cut a fixed epoch around every stimulus onset and
average it, so the contrast becomes a contrast of **stimulus responses** rather than of
whole tracks. A response locked to onsets is smeared out when the map spans every stimulus
at once.

One thing is genuinely different from the recording-axis variant, and it matters: **each
condition is epoched with its own onsets on its own segment.** The tracks keep their own
time bases here, so there is no single onset list covering both — `condition_onsets`
returns each condition's, local to its segment, and each split track is averaged against
its own. Both are cut on the same paradigm window from
[`AssrEpoch`](../../src/definitions/constants.py) via `iva_quality.onset_window`, taking
the shorter post-onset span of the two so the two averages share one epoch axis and can be
compared.

**No baseline subtraction**, as everywhere else: each track was z-scored over time, so the
pre-onset interval reads ≈ 0 by construction and structure there is a warning sign rather
than a response. Keeping the post-offset tail in view is what distinguishes a response that
stops with the stimulus from one that runs on. The prominent black lines are the stimulus
**onset** (dashed, `t = 0`) and its **offset** (dotted); the green line is 40 Hz.

Skipped for experiments without stimulus annotations, and when a condition's window is too
short to fit `MIN_ONSETS_FOR_EPOCH_AVERAGE` epochs.

In [ ]:
# Per condition: its own onsets, on its own split segment.
_epoch_info = {}
for name, arr in tf_by_condition.items():
    condition = next(c for c in paired.conditions if c.value == name)
    onsets = onsets_by_condition[condition]
    n_t = arr.shape[-1]
    if onsets is None or len(onsets) == 0:
        _epoch_info[name] = None
        continue
    onsets_in = np.asarray(onsets)[np.asarray(onsets) < n_t].astype(int)
    if onsets_in.size == 0:
        _epoch_info[name] = None
        continue
    pre, post = iva_quality.onset_window(onsets_in, n_t, sfreq)
    n_fitting = int(((onsets_in - pre >= 0) & (onsets_in + post <= n_t)).sum())
    _epoch_info[name] = (onsets_in, pre, post, n_fitting, n_t)
    print(
        f"{name:<12}: {len(onsets_in)} onset(s) in window, {n_fitting} epoch(s) fit, "
        f"window ({pre} pre, {post} post)"
    )

run_epoch_comparison = len(_epoch_info) > 0 and all(
    info is not None and info[3] >= MIN_ONSETS_FOR_EPOCH_AVERAGE
    for info in _epoch_info.values()
)

if not run_epoch_comparison:
    print(
        "\nStimulus-locked comparison skipped: at least one condition has fewer than "
        f"{MIN_ONSETS_FOR_EPOCH_AVERAGE} fitting epoch(s). Raise N_TIMES_SUBSET (or use "
        "an experiment with stimulus annotations)."
    )
else:
    # One epoch geometry for both conditions, so the two averages share an axis: the
    # shorter post-onset span wins (pre is a paradigm constant).
    EPOCH_PRE = min(info[1] for info in _epoch_info.values())
    EPOCH_POST = min(info[2] for info in _epoch_info.values())
    epoch_times = np.arange(-EPOCH_PRE, EPOCH_POST) / sfreq
    EPOCH_MARKS = [0.0, min(AssrEpoch.STIMULUS_DURATION_S, float(epoch_times[-1]))]

    onset_by_row = {}
    for name, arr in tf_by_condition.items():
        onsets_in = _epoch_info[name][0]
        # epoch_average works on any array whose LAST axis is time, so the global
        # (K, F, T_condition) maps go through unchanged.
        averaged, n_used = iva_quality.epoch_average(
            arr, onsets_in, EPOCH_PRE, EPOCH_POST
        )
        onset_by_row[name] = averaged
        print(f"{name:<12}: averaged {n_used} epoch(s) -> {averaged.shape}")
    onset_by_row[DIFFERENCE_ROW] = (
        onset_by_row[CONDITION_ROWS[1]] - onset_by_row[CONDITION_ROWS[0]]
    )

    print(
        f"\nEpoch window : {EPOCH_PRE + EPOCH_POST} samples ({EPOCH_PRE} pre, "
        f"{EPOCH_POST} post) = [{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s"
    )
    if EPOCH_POST < int(round(EPOCH_POST_S * sfreq)):
        print(
            f"  NOTE: post-onset span trimmed from {EPOCH_POST_S} s to "
            f"{EPOCH_POST / sfreq:.3f} s by the shortest inter-onset gap."
        )
    # The baseline should read ~0; structure there is a warning sign, not a response.
    _baseline = np.mean(
        [
            np.abs(onset_by_row[name][COMP_INDICES, :, :EPOCH_PRE]).mean()
            for name in CONDITION_ROWS
        ]
    )
    _response = np.mean(
        [
            np.abs(onset_by_row[name][COMP_INDICES, :, EPOCH_PRE:]).mean()
            for name in CONDITION_ROWS
        ]
    )
    print(
        f"Mean |baseline| / mean |post-onset| : {_baseline:.4g} / {_response:.4g}  "
        f"(ratio {_baseline / _response:.2f})"
    )

    fig_tf_onset = plot_global_tf_grid(
        onset_by_row,
        [*CONDITION_ROWS, DIFFERENCE_ROW],
        COMP_INDICES,
        FREQS,
        epoch_times,
        label=LABEL,
        title="Onset-averaged component TF maps, split by condition",
        freq_marks=TF_FREQ_MARKS,
        epoch_marks=EPOCH_MARKS,
        sign_note=SIGN_NOTE,
        save_path=(PLOTS_DIR / "condition_tf_maps_onset.png") if SAVE_PLOTS else None,
    )
    plt.show()
    plt.close("all")